In [ ]:
# ------- IMPORT LIBRARIES -------

import json
from bs4 import BeautifulSoup, Tag
import requests
import re
from concurrent.futures import ThreadPoolExecutor
import time
from datetime import datetime, timezone
import os
from urllib.parse import urlparse, unquote
from typing import Union, List, Optional
import sqlite3
from random import sample, choices
import pandas as pd
from openai import OpenAI
import random
from tqdm import tqdm
from functools import partial
from typing import Tuple
import logging



In [2]:
# --------------------- SETUP DATABASE ------------------

# set working directory
os.chdir('/home/schmi/projects/explain2me/data')

# Connect to DB (create if does not exist)
conn = sqlite3.connect(database="WikipediaOne.db")
cur = conn.cursor()

# create tables 'pages' and 'definitions'.
# 'definitions' contains the actual page content.
# 'pages' serves as a lookup table for 'definitions'
cur.execute("""
            CREATE TABLE IF NOT EXISTS pages (
			id INTEGER PRIMARY KEY,
			title TEXT UNIQUE NOT NULL,
			has_simple INTEGER DEFAULT 0,
			has_technical INTEGER DEFAULT 0,
			has_kids INTEGER DEFAULT 0
            );
            """)

cur.execute("""
            CREATE TABLE IF NOT EXISTS definitions (
			id INTEGER PRIMARY KEY,
			page_id INTEGER NOT NULL,
			kind TEXT CHECK(kind IN ('simple', 'technical', 'kids')),
			content TEXT NOT NULL,
			source TEXT,
			created_at TEXT,
			FOREIGN KEY (page_id) REFERENCES pages(id),
			UNIQUE (page_id, kind)
            );
			""")

conn.commit()
conn.close()

In [3]:
# ------------- HELPERS -------------
# (try to) convert simple wiki page url to its classic wiki page url 
# counterpart
# Goal -> have a normal wikipedia page for each simple wikipedia page 
#         already stored

def simple2normalwiki_url(simple_url):
    page_url_segment = simple_url.split("/")[-1]
    normalwiki_base = "https://en.wikipedia.org/wiki/"
    normalwiki_url = normalwiki_base + page_url_segment
    return normalwiki_url


#-----------------------------
# And conversely....

def normal2simplewiki_url(normal_url):
    page_url_segment = normal_url.split("/")[-1]
    simplewiki_base = "https://simple.wikipedia.org/wiki/"
    simplewiki_url = simplewiki_base + page_url_segment
    return simplewiki_url


# Normal Wikipedia page scraper

In [4]:
# ----------------------------------------
# 			  SCRAPING METHOD
# ----------------------------------------

# ---------------- CONFIG ----------------
IGNORE_CLASSES = {
    "sidebar-list", "navbar", "infobox", "toc",
    "thumb", "mw-default-size", "metadata"
}

STOP_SECTIONS = {
    "references", "external links", "see also", "notes", "further reading"
}


# ---------------- HELPERS ----------------
def clean_paragraph(el: Tag) -> str:
    """Clean paragraph text, preserving math as LaTeX."""

    # Remove citation markers
    for sup in el.find_all("sup"):
        sup.decompose()

    # Preserve math
    for math in el.find_all("math"):
        latex = math.get("alttext") or math.get_text(strip=True)
        latex = latex.strip()

        is_block = el.get_text(strip=True) == math.get_text(strip=True)
        math.replace_with(
            f"\n$$\n{latex}\n$$\n" if is_block else f"${latex}$"
        )

    # Lists
    if el.name in {"ul", "ol"}:
        lines = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            txt = clean_paragraph(li)
            if txt:
                lines.append(f"- {txt}" if el.name == "ul" else f"{i}) {txt}")
        return "\n".join(lines)

    # Text cleanup
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


def is_ignored(el: Tag) -> bool:
    """Ignore elements inside navboxes, infoboxes, thumbnails, TOC."""
    for parent in el.parents:
        classes = parent.get("class", [])
        if any(cls in IGNORE_CLASSES for cls in classes):
            return True
    return False



# ---------------- MAIN SCRAPER ----------------
def scrape_normal_wiki(url: str) -> dict:
    headers = {"User-Agent": "ReverseMentorBot/0.1"}
    
    try:
         res = requests.get(url, headers=headers, timeout=10)
         # Raise an HTTPError for 4xx/5xx responses (e.g., 404, 500, 429), ensuring failed HTTP responses are treated as errors.
         res.raise_for_status()
		
    except requests.RequestException as e:
        # catches all request-related failures: connection errors, timeouts, invalid URLs, and HTTP errors raised by raise_for_status()
		# i.e. network/environment-level failures, not parsing or scraper-logic errors.
        raise RuntimeError(f"Failed to fetch URL: {url}") from e


    soup = BeautifulSoup(res.text, "html.parser")


    content = soup.find("div", id="mw-content-text")
    
    if content is None:
        raise ValueError(f"Content div not found for {url}")

    sections = []
    intro = None
    current = None

    # Traverse in DOM order
    for el in content.find_all(
        ["p", "li", "dd", "ul", "ol", "h2", "h3", "h4", "h5"],
        recursive=True
    ):
        if is_ignored(el):
            continue

        # ---------- HEADINGS ----------
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in STOP_SECTIONS:
                break

            current = {"heading": heading, "paragraphs": []}
            sections.append(current)
            continue

        # ---------- CONTENT ----------
        text = clean_paragraph(el)
        if not text:
            continue

        if current is None:
            if intro is None:
                intro = {"heading": "Introduction", "paragraphs": []}
                sections.insert(0, intro)
            intro["paragraphs"].append(text)
        else:
            current["paragraphs"].append(text)

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else None
    

	# --------- CATEGORY DATA ----------
    categories = []
    category_urls = []

    catlinks = soup.select("#mw-normal-catlinks ul li a")
    for cat in catlinks:
        categories.append(cat.get_text(strip=True))
        href = cat.get("href")
        if href and href.startswith("/wiki/"):
            category_urls.append("https://en.wikipedia.org" + href)



    return {
        "url": url,
        "title": title,
        "sections": sections,
        "categories": categories,
        "category_urls": category_urls,
    }



In [5]:
test_scrape_normal_wiki1 = scrape_normal_wiki(url='https://en.wikipedia.org/wiki/Graph_database')
test_scrape_normal_wiki1

{'url': 'https://en.wikipedia.org/wiki/Graph_database',
 'title': 'Graph database',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['A graph database ( GDB ) is a database that uses graph structures for semantic queries with nodes, edges, and properties to represent and store data. A key concept of the system is the graph (or edge or relationship). The graph relates the data items in the store to a collection of nodes and edges, the edges representing the relationships between the nodes. The relationships allow data in the store to be linked together directly and, in many cases, retrieved with one operation. Graph databases hold the relationships between data as a priority. Querying relationships is fast because they are perpetually stored in the database. Relationships can be intuitively visualized using graph databases, making them useful for heavily inter-connected data.',
    'Graph databases are commonly referred to as a NoSQL database. Graph databases are similar to 1

# Simple wiki page scraper

In [ ]:

# ---------------- HELPERS ----------------


def clean_paragraph(el: Tag):
    """
    Clean paragraph text, preserving formulas as LaTeX.
    Handles <p>, <li>, <dd>, <ul>, <ol> elements.
    """

    # Remove citation superscripts
    for sup in el.find_all("sup"):
        sup.decompose()

    # Replace <math> elements with LaTeX
    for math in el.find_all("math"):
        latex = math.get("alttext") or "".join(math.strings).strip()
        math.replace_with(f"${latex.strip()}$")

    # Handle lists
    if el.name in ["ul", "ol"]:
        items = []
        for i, li in enumerate(el.find_all("li", recursive=False), start=1):
            li_text = clean_paragraph(li)
            if li_text:
                items.append(f"- {li_text}" if el.name == "ul" else f"{i}) {li_text}")
        return "\n".join(items)

    # Handle list items and description items
    if el.name in ["li", "dd"]:
        parts = []
        for child in el.children:
            if isinstance(child, Tag):
                parts.append(clean_paragraph(child))
            else:
                parts.append(str(child))
        text = " ".join(filter(None, parts))
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s+([.,;:!?])", r"\1", text)
        return text.strip()

    # Default text
    text = el.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text.strip()


# --- Recursive content iterator ---
def iter_content_elements(el):
    """
    Yield all relevant content elements in document order.
    Skip navboxes, tables, scripts, styles.
    """
    for child in el.children:
        if not isinstance(child, Tag):
            continue

        if child.name in ["table", "script", "style"]:
            continue

        # Skip sideboxes, navboxes, metadata
        if child.name == "div":
            classes = child.get("class") or []
            if any(c in ["navbox", "vertical-navbox", "metadata", "mbox"] for c in classes):
                continue
            yield from iter_content_elements(child)
            continue
        
        # Skip geo/coordinates spans
        if child.name == "span" and any(c in ["geo", "coordinates"] for c in (child.get("class") or [])):
            continue

		# Recursively yield from spans (other inline containers)
        if child.name == "span":
            yield from iter_content_elements(child)
            continue

        # Yield headings and paragraph-like content
        if child.name in ["p", "ul", "ol", "dd"] + [f"h{i}" for i in range(2, 7)]:
            yield child
        else:
            yield from iter_content_elements(child)





In [7]:

# ---------------- MAIN SCRAPER ----------------

def scrape_simple_wiki(url):
    """
    Scrapes a Simple Wikipedia page and returns structured article data.
    Handles redirects, missing content, and network errors.
    """
    
    headers = {
        "User-Agent": "ReverseMentorBot/0.1 (https://yourdomain.com/contact)"
    }

    stop_sections = {
        "references",
        "other websites",
        "related pages",
        "further reading",
        "external links",
        "see also",
    }

    # --- Fetch page with network error handling ---
    try:
        res = requests.get(url, headers=headers, timeout=10)
        # Raises for 4xx/5xx responses
        res.raise_for_status()
    except requests.RequestException as e:
        # Network/environment-level failure: connection, timeout, HTTP error, invalid URL
        raise RuntimeError(f"Failed to fetch Simple Wikipedia URL: {url}") from e

    soup = BeautifulSoup(res.text, "html.parser")

    # --- Handle redirects ---
    redirect_div = soup.find("div", class_="redirectMsg")
    if redirect_div and redirect_div.find("a"):
        redirect_url = "https://simple.wikipedia.org" + redirect_div.find("a")["href"]
        return scrape_simple_wiki(redirect_url)


    # --- Main content ---
    content = soup.find("div", class_="mw-parser-output")
    if content is None:
        # Page layout changed or empty page
        raise ValueError(f"Main content not found for {url}")

    # --- Article title with fallback ---
    title_tag = soup.find("h1", id="firstHeading")
    if title_tag:
        title = title_tag.get_text(strip=True)
    else:
        # fallback: use last segment of URL
        title = url.split("/")[-1].replace("_", " ")

    article_data = {
        "url": url,
        "title": title,
        "sections": [],
        "categories": [],
        "category_urls": [],
    }

    # --- Introduction section ---
    intro_section = {"heading": "Introduction", "paragraphs": []}
    current_section = intro_section

    # --- Walk content recursively ---
    for el in iter_content_elements(content):
        # Headings start new sections
        if el.name.startswith("h"):
            heading = el.get_text(" ", strip=True).replace("[edit]", "")
            if heading.lower() in stop_sections:
                break
            if intro_section["paragraphs"] and intro_section not in article_data["sections"]:
                article_data["sections"].append(intro_section)
            current_section = {"heading": heading, "paragraphs": []}
            article_data["sections"].append(current_section)
            continue

        # Paragraph-like content
        if el.name in ["p", "ul", "ol", "dd"]:
            text = clean_paragraph(el)
            if text:
                current_section["paragraphs"].append(text)

    # --- Ensure intro is included if it has paragraphs ---
    if intro_section["paragraphs"] and intro_section not in article_data["sections"]:
        article_data["sections"].insert(0, intro_section)

    # --- Categories ---
    catlinks = soup.select("#mw-normal-catlinks ul li a")
    for cat in catlinks:
        article_data["categories"].append(cat.get_text(strip=True))
        href = cat.get("href")
        if href and href.startswith("/wiki/"):
            article_data["category_urls"].append("https://simple.wikipedia.org" + href)

    # --- Ensure at least one section exists ---
    if not article_data["sections"]:
        raise ValueError(f"No sections found for {url}")

    return article_data


In [8]:
test_scrape_simple_wiki_redirect1 = scrape_simple_wiki(url='https://simple.wikipedia.org/wiki/USA') # redirect to 'https://simple.wikipedia.org/wiki/United_States
test_scrape_simple_wiki_redirect1

{'url': 'https://simple.wikipedia.org/wiki/USA',
 'title': 'United States',
 'sections': [{'heading': 'Introduction',
   'paragraphs': ['The United States of America ( USA ), also known as the United States ( U.S. or US ) or colloquially as America, is a country that is mainly in North America. It is made of 50 states, a federal district ( Washington, D.C., the District of Columbia), and some other territories and insular areas. Forty-eight of the states are connected ( Contiguous United States ), and they are bordered by Canada to the north and Mexico to the south. The state of Alaska is in the northwestern area of the continent near Asia ( Russia ) and is separated from the other 48 states by Canada making it an exclave. Alaska is bordered by Canada to its east. The state of Hawaii is a set of islands in the Pacific located within Polynesia and is about 2,200 miles (3,500 kilometers) from the continent. The capital city is Washington, D.C. and the largest city by population is New Yo

In [9]:
# -------- Get Wiki pages URLs from a Wiki Category URL --------------
# Used for scraping all pages in a category instead of scraping them individually.
# Works for both simple and normal wiki category pages.

def get_category_pages(category_url):

    headers = {
        "User-Agent": "YourBot/1.0 (https://example.com/contact)"
    }
    
    try:
         res = requests.get(category_url, headers=headers, timeout=10)
         # Raise an HTTPError for 4xx/5xx responses (e.g., 404, 500, 429), ensuring failed HTTP responses are treated as errors.
         res.raise_for_status()
		
    except requests.RequestException as e:
        # catches all request-related failures: connection errors, timeouts, invalid URLs, and HTTP errors raised by raise_for_status()
		# i.e. network/environment-level failures, not parsing or scraper-logic errors.
        raise RuntimeError(f"Failed to fetch category URL: {category_url}") from e
		
    
    soup = BeautifulSoup(res.text, "html.parser")

    # Extract category name from URL
    url_parse = urlparse(category_url)
    path = url_parse.path
    category_name = path.split(":")[-1]

    base = url_parse.scheme + "://" + url_parse.netloc # https://simple.wikipedia.org
    pages = []

    for li in soup.select("#mw-pages li a"):
        href = li.get("href")
        title = li.get_text(strip=True)
        if "Template:" in title:
            continue
        if href and href.startswith("/wiki/"):
            pages.append({
                "title": title,
                "url": base + href
            })
            
    if not pages:
            raise ValueError(f"No pages found in category {category_name} at {category_url}")

    return {
        "categories": category_name,
        "category_urls": category_url,
        "pages": pages
    }

In [10]:
# test get_category_pages()
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/wiki/Category:Movie_producers_from_New_York_City')

# gets only the 200 first pages in the category (wiki category page structure - category has multiple pages if more than 200 pages in the category)
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/wiki/Category:Living_people')
# but works if subsecant pages url is provided:
simple_category_test = get_category_pages(category_url='https://simple.wikipedia.org/w/index.php?title=Category:Living_people&pagefrom=Abrines+Redondo%2C+Alejandro%0AAlejandro+Abrines+Redondo#mw-pages')

# works also for normal wiki categories:
normal_category_test = get_category_pages(category_url='https://en.wikipedia.org/wiki/Category:Database_models')

len(simple_category_test['pages']), simple_category_test['pages']

(200,
 [{'title': 'Alejandro Abrines Redondo',
   'url': 'https://simple.wikipedia.org/wiki/Alejandro_Abrines_Redondo'},
  {'title': 'Anne-Ségolène Abscheidt',
   'url': 'https://simple.wikipedia.org/wiki/Anne-S%C3%A9gol%C3%A8ne_Abscheidt'},
  {'title': 'Mohammad Abshak',
   'url': 'https://simple.wikipedia.org/wiki/Mohammad_Abshak'},
  {'title': 'Mehdi Abtahi',
   'url': 'https://simple.wikipedia.org/wiki/Mehdi_Abtahi'},
  {'title': 'Najmeh Abtin',
   'url': 'https://simple.wikipedia.org/wiki/Najmeh_Abtin'},
  {'title': 'Abu Hafs al-Hashimi al-Qurashi',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Hafs_al-Hashimi_al-Qurashi'},
  {'title': 'Abu Haider',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Haider'},
  {'title': 'Eva Abu Halaweh',
   'url': 'https://simple.wikipedia.org/wiki/Eva_Abu_Halaweh'},
  {'title': 'Abu Hamza al-Masri',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Hamza_al-Masri'},
  {'title': 'Abu Khaled',
   'url': 'https://simple.wikipedia.org/wiki/Abu_Khal

In [11]:

# --------- Check if page is already in DB ----------------------------
# ------------- or should be scraped ----------------------------------


def page_needs_scraping(url: str, db_path: str) -> bool:
    """
    Returns True if the page (simple or technical) is NOT yet stored.
    Infers everything from the URL.
    """

    # Extract title
    path = urlparse(url).path
    if "/wiki/" not in path:
        return False  # Not a valid wiki page

    title = unquote(path.split("/wiki/")[-1])

    # Determine which indicator column to check
    if "simple.wikipedia.org" in url:
        indicator_col = "has_simple"
    elif "wikipedia.org" in url:
        indicator_col = "has_technical"
    else:
        return False  # Not supported domain

    # Open connection (thread-safe pattern)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    try:
        cur.execute(
            f"SELECT {indicator_col} FROM pages WHERE title = ?",
            (title,)
        )
        row = cur.fetchone()
    finally:
        conn.close()

    if row is None:
        # Page not in DB at all → needs scraping
        return True

    # If indicator is 0 → needs scraping
    return row[0] == 0

In [12]:
# test
page_needs_scraping(url='https://simple.wikipedia.org/wiki/Markov_chain', db_path="WikipediaOne.db")
page_needs_scraping(url='https://simple.wikipedia.org/wiki/Stochastic_process', db_path="WikipediaOne.db")

True

In [13]:
# ------------ STORAGE FCT TO DB ---------------

def store_page(article_data: dict, conn, cur):
    """
    Stores a scraped Wikipedia article into DB.
    (Updates both tables 'pages' and 'definitions')
    
    article_data = {
        "url": str,
        "title": str,
        "sections": list,
        ...
    }
    """

    url = article_data["url"]
    title = article_data["title"]
    sections = article_data.get("sections", [])
    content = json.dumps(sections)  # store sections as JSON string

    # Determine kind from URL
    kind = "simple" if "simple.wikipedia.org" in url else "technical"

    source = url
    created_at = datetime.now(timezone.utc).isoformat()

    # Ensure page exists (redondance of 'OR IGNORE' as scrape_wikipedia already handles page uniqueness)
    cur.execute(
        "INSERT OR IGNORE INTO pages (title) VALUES (?);",
        (title,)
    )

    # Get page_id
    cur.execute("SELECT id FROM pages WHERE title = ?", (title,))
    page_id = cur.fetchone()[0]

    # Insert definition or update existing 
	# to correct: (scrape_wikipedia() prevents updates though as checks for existance of title first)
    cur.execute(
        """
        INSERT OR REPLACE INTO definitions
        (page_id, kind, content, source, created_at)
        VALUES (?, ?, ?, ?, ?);
        """,
        (page_id, kind, content, source, created_at)
    )

    # Update indicator & commit
    cur.execute(f"UPDATE pages SET has_{kind} = 1 WHERE id = ?", (page_id,))

    conn.commit()


In [14]:
# ---- FRAMEWORK FUNCTION - MAIN SCRAPING FUNCTION ----

def scrape_wikipedia(
        urls: Union[str, List[str]], 
        db_path: Optional[str] = None, 
        cat_workers: int = 5
    ) -> List[dict]:
    """
    General Wikipedia scraper framework.
    
    Parameters
    ----------
    urls : str or List[str]
        Wikipedia page(s) or category URL(s) to scrape.
    db_path : str, optional
        Path to SQLite DB to store results. If None, results are not stored.
    cat_workers : int
        Number of parallel threads for scraping.

    Returns
    -------
    List[dict]
        List of successfully scraped page results.
    
    Notes:
    - Accepts single URL or list of URLs (pages or categories)
    - Automatically detects:
        - Simple vs normal Wikipedia
        - Category vs single page
    - Expands categories to individual page URLs
    - Scrapes pages in parallel using ThreadPoolExecutor
    - Directs to the appropriate scraper function
    """

    if isinstance(urls, str):
        urls = [urls]

    # Expand all category URLs
    expanded_urls = []
    for url in urls:
        if "/wiki/Category:" in url:
            categories = get_category_pages(url)
            pages_urls = [page.get('url', None) for page in categories['pages']]
            expanded_urls.extend(pages_urls)
        else:
            expanded_urls.append(url)
    
    # Decide which need scraping (if db_path provided, don't scrape if already in db)     
    urls_to_scrape = []
    
    for url in expanded_urls:
        if db_path is None:
            urls_to_scrape.append(url)
        else:
            if page_needs_scraping(url, db_path=db_path):
                urls_to_scrape.append(url)
     
    if not urls_to_scrape:
        print("All pages already in DB. Nothing to scrape.")
        return []
    
    
	# Determine which scraper to use for each URL
    def scrape_dispatcher(url: str):
        conn = None
        cur = None
        if db_path:
            conn = sqlite3.connect(db_path)
            cur = conn.cursor()

        try:
            if "simple.wikipedia.org" in url:
                result = scrape_simple_wiki(url)
            else:
                result = scrape_normal_wiki(url)

            if db_path and result:
                store_page(result, conn, cur)

            return result

        except RuntimeError:
            print(f"[WARNING] Skipping URL due to fetch error: {url}")
            return None

        finally: 
            if conn: conn.close()


    # Parallel scraping
    results = []
    with ThreadPoolExecutor(max_workers=cat_workers) as executor:
        futures = [executor.submit(scrape_dispatcher, u) for u in urls_to_scrape]
        for f in futures:
            res = f.result()
            results.append(res)

    return results

In [15]:
# test
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Data')
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Markov_chain', db_path='WikipediaOne.db')
scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Stochastic_process', db_path='WikipediaOne.db')

test1 = ['https://simple.wikipedia.org/wiki/Stochastic_process',
         'https://simple.wikipedia.org/wiki/Category:Statistics',]

test1_res = scrape_wikipedia(urls=test1, db_path='WikipediaOne.db')
len(test1_res), test1_res

test2 = ['https://en.wikipedia.org/wiki/Category:Database_models',
         'https://en.wikipedia.org/wiki/Heterogeneous_database_system',]

test2_res = scrape_wikipedia(urls=test2, db_path='WikipediaOne.db')
len(test2_res), test2_res

scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Blockchain', db_path='WikipediaOne.db')

scrape_wikipedia(urls='https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', db_path='WikipediaOne.db')

scrape_wikipedia(urls='https://en.wikipedia.org/wiki/Discrete_time_and_continuous_time', db_path='WikipediaOne.db')

scrape_wikipedia(urls='https://simple.wikipedia.org/wiki/Allele', db_path='WikipediaOne.db')

[{'url': 'https://simple.wikipedia.org/wiki/Allele',
  'title': 'Allele',
  'sections': [{'heading': 'Introduction',
    'paragraphs': ['An allele is a form of a gene at a particular position ( locus ) on a chromosome. It is the bit of coding DNA at that place.',
     'Typical plants and animals have two sets of chromosomes, one set inherited from each parent. These organisms are called diploid. Since such organisms have two sets of chromosomes, they have (except on the sex chromosomes ) two alleles at each gene locus.',
     'If the two alleles are identical, the individual is called a homozygote and is said to be homozygous. If instead the two alleles are different, the individual is a heterozygote and is heterozygous.']},
   {'heading': 'Dominance',
    'paragraphs': ['In a heterozygote the effect of one allele may completely ‘mask’ the other. That is, the phenotype produced by the two alleles in heterozygous combination is identical to that produced by one of the two homozygous gen

In [ ]:


# put in main.py

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(funcName)s:%(lineno)d | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("toy_test_log.log")
    ],
    force=True
)


In [ ]:

# ------------------------ BACKFILL DB ------------------------
# ----- with existing simple/technical page counterparts ------




logger = logging.getLogger(__name__)

def backfill_DB(
        db_path: str,
        cat_workers: int = 5
    ) -> List[Optional[dict]]:
    """
    Checks DB for pages where:
        - has_simple = 0
        - has_technical = 0
    
    Attempts to construct the missing URL version (works only if direct mapping between urls) and scrape it.
    
    Returns list of successfully scraped pages.
    """
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        
		# Fetch all pages
        cur.execute("""
            SELECT id, title, has_simple, has_technical
            FROM pages
            WHERE has_simple = 0 OR has_technical = 0;
            """)
        
        rows = cur.fetchall()
        conn.close()
        
        logger.info("Found %d pages missing wiki versions.", len(rows))
        
    except sqlite3.Error as e:
        logger.exception("Database error during backfill_DB. db_path='%s'",
                         db_path)
        raise
    
    urls_to_scrape = []
    
    for _, title, has_simple, has_technical in rows:
        if has_simple == 0:
            # construct canonical technical or simple URLs and add to scraping list
            simple_url = f"https://simple.wikipedia.org/wiki/{title.replace(' ', '_')}"
            urls_to_scrape.append(simple_url)
        if has_technical == 0:
            normal_url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
            urls_to_scrape.append(normal_url)
      

    if not urls_to_scrape:
        logger.info("No missing wiki versions found in  db_path='%s'.", db_path)
        return []

    logger.info(
        "Attempting to scrape %d missing versions...",
        len(urls_to_scrape)
    )

    try:
        results = scrape_wikipedia(
            urls=urls_to_scrape,
            db_path=db_path,
            cat_workers=cat_workers
        )
        logger.info("Scraping completed.")
        return results # some elem in list results might be None but scrape_wikipedia() does not store them

    except Exception:
        logger.exception("Scraping failed in backfill_DB")
        raise




In [18]:
test_backfill_DB1 = backfill_DB(db_path='WikipediaOne.db')
len(test_backfill_DB1), test_backfill_DB1

2026-02-17 10:30:08,553 | INFO | __main__ | backfill_DB:36 | Found 115 pages missing wiki versions.
2026-02-17 10:30:08,555 | INFO | __main__ | backfill_DB:59 | Attempting to scrape 115 missing versions...


[WARNING] Skipping URL due to fetch error: https://en.wikipedia.org/wiki/Inference_(statistics)
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Database_model
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Data_integration
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Component-oriented_database
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Array_DBMS
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Data_orientation
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Entity–attribute–value_model
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Flat-file_database
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Graph_database
[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Heterogeneous_database_system
[WARNING] Skipping URL due to 

2026-02-17 10:30:38,963 | INFO | __main__ | backfill_DB:70 | Scraping completed.


[WARNING] Skipping URL due to fetch error: https://simple.wikipedia.org/wiki/Discrete_time_and_continuous_time


(115,
 [{'url': 'https://en.wikipedia.org/wiki/Markov_chain',
   'title': 'Markov chain',
   'sections': [{'heading': 'Introduction',
     'paragraphs': ['- Probability - Axioms\n- Determinism - System\n- Indeterminism\n- Randomness',
      'Probability - Axioms',
      '- Axioms',
      'Axioms',
      'Determinism - System',
      '- System',
      'System',
      'Indeterminism',
      'Randomness',
      '- Probability space\n- Sample space\n- Event - Collectively exhaustive events - Elementary event - Mutual exclusivity - Outcome - Singleton\n- Experiment - Bernoulli trial\n- Probability distribution - Bernoulli distribution - Binomial distribution - Exponential distribution - Normal distribution - Pareto distribution - Poisson distribution\n- Probability measure\n- Random variable - Bernoulli process - Continuous or discrete - Expected value - Variance - Markov chain - Observed value - Random walk - Stochastic process',
      'Probability space',
      'Sample space',
      'Even

In [19]:
[elem.get('title','') for elem in test_backfill_DB1 if elem != None]

['Markov chain',
 'Stochastic process',
 'Regression toward the mean',
 'Analytics',
 'Average',
 'Autoregressive model',
 "Bessel's correction",
 'Basic reproduction number',
 'Bose–Einstein statistics',
 'Biostatistics',
 'Central limit theorem',
 'Central tendency',
 'Cluster analysis',
 'Consumer price index',
 'Confidence interval',
 'Contour line',
 'Correlation',
 'Demography',
 'Coupling constant',
 'Data',
 'Degrees of freedom (statistics)',
 'Dunn index',
 'Efficiency (statistics)',
 'Expected value',
 'Descriptive statistics',
 'Estimator',
 'Dependent and independent variables',
 'Frequency (statistics)',
 'Failure rate',
 'Frequentist probability',
 "Gambler's fallacy",
 'Games played',
 'Generative model',
 'Geometric distribution',
 'Geometric mean',
 'Graph',
 'Gini coefficient',
 'Grouped data',
 'Infant mortality',
 'Histogram',
 'Harmonic mean',
 'Independence (probability theory)',
 'Infinitesimal model',
 'Input–output model',
 'Lady tasting tea',
 'Interquartile r

In [27]:
# --------------------------------------- FOR UNIT TEST ONLY ---------------------------------------
# --------------------------------------- FOR UNIT TEST ONLY ---------------------------------------
# --------------------------------------- FOR UNIT TEST ONLY ---------------------------------------


client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

MAX_INPUT_TOKENS = 16384 # model's (Llama-3.1-8B-Instruct) context length (16384 tokens).

data_path = os.getcwd()
db_name = 'WikipediaOne.db'
db_path = data_path + '/' + db_name


# get necessary data 
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# fetching logic: use first simple definitions (lower nb token + less complex text for a simple model)
cur.execute("""
		SELECT
			p.id,
			p.title,
			CASE
				WHEN p.has_simple = 1 THEN s.content
				ELSE t.content
			END AS selected_content
		FROM pages p
		LEFT JOIN definitions s ON p.id = s.page_id AND s.kind = 'simple'
		LEFT JOIN definitions t ON p.id = t.page_id AND t.kind = 'technical'
		WHERE p.has_simple = 1 OR p.has_technical = 1;
	""")

data4gen = cur.fetchall()
conn.close()

# --------------------------------------- FOR UNIT TEST ONLY ---------------------------------------
# --------------------------------------- FOR UNIT TEST ONLY ---------------------------------------
# --------------------------------------- FOR UNIT TEST ONLY ---------------------------------------


In [28]:
# ----------------- Definition for Kids - Generation ---------------


# generation of definitions for kids using a LLM, based on scraped wikipedia page content (use simple definition if available)
def generate_kids_definition(
    page_tuple: Tuple[int, str, str],
    client: OpenAI,
    max_input_tokens: int=16384,
    max_retries: int = 3,
    base_delay: float = 1.0,) -> Tuple[int, str] | None:
    
    """
    page_tuple: (page_id, title, content)
    """
    
    page_id, title, description = page_tuple
    description = (description or "")[:max_input_tokens]

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="meta-llama/Llama-3.1-8B-Instruct:novita",
                messages=[
                    {"role": "system",
                        "content": (
                            "Explain concepts clearly for children around 10 years old. "
                            "Use simple words, short sentences, and concrete examples. "
                            "Avoid technical terms unless explained. "
                            "No titles or meta commentary. Output only the explanation."
                        ),
                    },
                    {"role": "user",
                        "content": f"Explain this for a 10-year-old:\n\n"
                                   f"topic: {title}\n"
                                   f"description: {description}",
                    },
                ],
                
                max_completion_tokens=500,
            )

            kids_definition = completion.choices[0].message.content
            return (page_id, kids_definition)

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"[FAILED] page \"{title}\" after {max_retries} attempts: {e}")
                return None

            # exponential backoff
            sleep_time = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            time.sleep(sleep_time)
            

#test
generate_kids_definition(data4gen[3], client=client)




2026-02-17 10:30:43,316 | INFO | httpx | _send_single_request:1025 | HTTP Request: POST https://router.huggingface.co/v1/chat/completions "HTTP/1.1 200 OK"


(4,
 'Imagine you have a lemonade stand. You want to know how to make more money and sell more lemonade. But, you don\'t know how to do it. That\'s where analytics comes in.\n\nAnalytics is like a super cool tool that helps you understand your lemonade stand. It looks at all the numbers and information about how many cups of lemonade you sell, how much money you make, and when you sell it.\n\nAnalytics takes all that raw information and turns it into helpful ideas. For example, it might say, "Hey, I noticed that you sell more lemonade on Fridays than on Mondays." Or, "You make more money if you sell lemonade for 50 cents per cup instead of 75 cents."\n\nAnalytics is like having a friend who helps you make sense of all the numbers and information. It helps you make better decisions, like what time to open your lemonade stand or how much lemonade to make.')

In [29]:
# ----- fetching data from db for kid definition generation ------

# Fetching function to fetch all pages content to feed the llm 

def fetch4kidgen(db_path: str):
	conn = sqlite3.connect(db_path)
	cur = conn.cursor()

	# fetching logic: use first simple definitions (lower nb token + less complex text for a simple model)
	cur.execute("""
		SELECT
			p.id,
			p.title,
			CASE
				WHEN p.has_simple = 1 THEN s.content
				ELSE t.content
			END AS selected_content
		FROM pages p
		LEFT JOIN definitions s ON p.id = s.page_id AND s.kind = 'simple'
		LEFT JOIN definitions t ON p.id = t.page_id AND t.kind = 'technical'
		WHERE (p.has_simple = 1 OR p.has_technical = 1) AND p.has_kids = 0;
	""")

	data4gen = cur.fetchall()
	conn.close()

	return data4gen

# test
data4gen = fetch4kidgen(db_path=db_path)
data4gen[5:7]


[(6,
  'Autoregressive model',
  '[{"heading": "Introduction", "paragraphs": ["An autoregressive model is a kind of statistical model. Like all statistics models, the idea is to describe a random process. In an autoregressive model, the output value depends linearly on one of the previous values of the model, plus a random variable, which describes that there is some randomness in the outcome."]}]'),
 (7,
  "Bessel's correction",
  '[{"heading": "Introduction", "paragraphs": ["Bessel\'s correction has high importance in calculating standard deviation. As per Bessel\'s correction, we should consider n-1 separation while calculating standard deviation of sampled data."]}]')]

In [30]:
# STORING FUNCTION

def store_kids(kid_defs: list, db_path: str) -> None:
	"""
    kid_defs: List
			List of tuples. tuples: (page_id: int, kids_definition: str)
    """
	conn = sqlite3.connect(db_path)
	cur = conn.cursor()
	for page_id, kid_def in kid_defs:
		if kid_def:
			cur.execute("""
				INSERT OR REPLACE INTO definitions
				(page_id, kind, content, source, created_at)
				VALUES (?, 'kids', ?, 'LLM_generated', ?)
			""", (page_id, kid_def, datetime.now(timezone.utc).isoformat()))
			cur.execute("UPDATE pages SET has_kids = 1 WHERE id=?", (page_id,))
	conn.commit()
	conn.close()

In [31]:
# FULL FUNCTION

def generate_n_populate_kid_def(
    db_path: str,
    client: OpenAI,
    max_input_tokens: int,
    max_workers: int = 5,
    ) -> None:

    data4gen = fetch4kidgen(db_path=db_path)
    
    if not data4gen:
        print("No pages to generate kids definitions for - DB might be already fully populated")
        return

    worker = partial(
        generate_kids_definition,
        client=client,
        max_input_tokens=max_input_tokens,
    )

    print(f"Generating kids definitions for {len(data4gen)} pages...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(
            tqdm(
                executor.map(worker, data4gen),
                total=len(data4gen),
                desc="Generating"
            )
        )

    # Remove failed generations
    results = [r for r in results if r is not None]

    print(f"Storing {len(results)} successful generations...")

    store_kids(kid_defs=results, db_path=db_path)

    print("Done.")


In [49]:
# Config
data_path = os.getcwd()
db_name = 'WikipediaOne.db'
db_path = data_path + '/' + db_name

MAX_INPUT_TOKENS = 16384 # model's (Llama-3.1-8B-Instruct) context length (16384 tokens).

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)


# Main - Generate kid definition for all wiki title stored in db
generate_n_populate_kid_def(db_path=db_path, client=client, max_workers=5, max_input_tokens=MAX_INPUT_TOKENS)

No pages to generate kids definitions for - DB might be already fully populated


In [47]:
# test

conn = sqlite3.connect("WikipediaOne.db")

df = pd.read_sql_query(
    """
    SELECT p.title, d.*
    FROM pages p
    JOIN definitions d ON p.id = d.page_id
    WHERE d.kind='kids';
    """,
    conn,
    index_col='id'  # <- use the 'id' column from definitions as index
)

conn.close()

df.sort_values('page_id')



,title,page_id,kind,content,source,created_at
id,,,,,,
223,Markov chain,1,kids,Imagine you have a friend who likes to eat thr...,LLM_generated,2026-02-17T09:32:52.686979+00:00
224,Stochastic process,2,kids,Imagine you're trying to predict the weather. ...,LLM_generated,2026-02-17T09:32:52.687418+00:00
225,Regression toward the mean,3,kids,"Imagine you have a friend who is really, reall...",LLM_generated,2026-02-17T09:32:52.687473+00:00
226,Analytics,4,kids,Imagine you're a manager of a lemonade stand. ...,LLM_generated,2026-02-17T09:32:52.687488+00:00
227,Average,5,kids,Imagine you and your friends collected differe...,LLM_generated,2026-02-17T09:32:52.687499+00:00
...,...,...,...,...,...,...
300,Maximum likelihood estimation,123,kids,"Imagine you have a big jar of cookies, and you...",LLM_generated,2026-02-17T09:37:25.437387+00:00
301,Statistical population,124,kids,Imagine you're doing a science project and you...,LLM_generated,2026-02-17T09:37:25.437398+00:00
302,Sample size determination,125,kids,Imagine you want to know how many people in yo...,LLM_generated,2026-02-17T09:37:25.437407+00:00


In [48]:
len(df)

127